In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

df=pd.read_excel('../data/raw/hand_made_original_data.xlsx')

## 1. Original Data Checking

In [2]:
print('=========================================')
print('1. check original data')
print('=========================================\n')
print('✅ sample display\n')
print(df.head(5))
print('\n✅ info statistic\n')
print(df.info())
print('\n✅ basic calculation\n')
print(df.describe())
print('\n✅ basic calculation\n')
print(df.columns.to_list())

1. check original data

✅ sample display

               上市时间 交付时间  新车状态 车厂背景    车厂            车型    车型级别 动力类型  \
0            2021/9  NaN  新车上市  新势力  小鹏汽车          小鹏P5    紧凑型车  NaN   
1            2021/1  NaN  新车上市  NaN  合资大众  大众ID.4 Crozz  紧凑型SUV  NaN   
2  2021/1（其他配置陆续上市）  NaN  新车上市  NaN   特斯拉       Model Y   中型SUV  NaN   
3            2021/2  NaN  新车上市  NaN  上汽飞凡     荣威MarvelR   中型SUV  NaN   
4            2021/2  NaN  新车上市  NaN  东风本田        本田CR-V  紧凑型SUV  NaN   

         补贴后售价 续航（CLTC）                   尺寸  车身结构      电量KWh     电池类型   电池厂  \
0    16.3-22.9  460-600  4808*1840*1520*2768  四门五座  55.9-66.2  三元/磷酸铁锂  亿纬锂能   
1   20.83-28.3  402-555  4612*1852*1640*2765  五门五座  57.3-83.4       三元   NaN   
2    29.2-34.8  526-640  4750*1921*1624*2890  五门五座    60-78.4  三元/磷酸铁锂   NaN   
3  21.98-26.18  460-505  4674*1919*1618*2800  五门五座       69.9       三元   NaN   
4  27.38-29.98       85  4694*1861*1679*2660  五门五座         16    锂电子电池   NaN   

    竞品 Unnamed: 16  
0  NaN         NaN  
1 

## 2. Column Mapping

In [3]:
col_orig=df.columns.to_list()
col_proj=['launch_date', 'delivery_date', 'vehicle_status', 
          'maker_type', 'brand', 'model', 'vehicle_class', 
          'powertrain', 'subsidized_price', 'range_cltc', 'dimensions', 
          'body_structure', 'battery_capacity_kwh', 'battery_chemistry', 
          'battery_supplier', 'competitors', 'to_drop']
col_mapping=dict(zip(col_orig,col_proj))
df.rename(columns=col_mapping,inplace=True)
df.drop(columns=['to_drop','delivery_date','vehicle_status'],inplace=True)

In [5]:
print('=========================================')
print('2. Column Mapping')
print('=========================================\n')
print('\n✅ mapping\n')
print(col_mapping)
print('\n✅ modified columns\n')
print(df.columns)

2. Column Mapping


✅ mapping

{'上市时间': 'launch_date', '交付时间': 'delivery_date', '新车状态': 'vehicle_status', '车厂背景': 'maker_type', '车厂': 'brand', '车型': 'model', '车型级别': 'vehicle_class', '动力类型': 'powertrain', '补贴后售价': 'subsidized_price', '续航（CLTC）': 'range_cltc', '尺寸': 'dimensions', '车身结构': 'body_structure', '电量KWh': 'battery_capacity_kwh', '电池类型': 'battery_chemistry', '电池厂': 'battery_supplier', '竞品': 'competitors', 'Unnamed: 16': 'to_drop'}

✅ modified columns

Index(['launch_date', 'maker_type', 'brand', 'model', 'vehicle_class',
       'powertrain', 'subsidized_price', 'range_cltc', 'dimensions',
       'body_structure', 'battery_capacity_kwh', 'battery_chemistry',
       'battery_supplier', 'competitors'],
      dtype='object')


## 3. Data Cleansing

In [6]:
print('=========================================')
print('3. unique value of columns')
print('=========================================\n')
for col in df.columns:
    print(f'\n✅ {col}\n')
    print(df[f'{col}'].unique())

3. unique value of columns


✅ launch_date

['2021/9' '2021/1' '2021/1（其他配置陆续上市）' '2021/2' '2021/3' '2021/4-5-6'
 '2021/4' '2021/5' '2021/4 & 2021/6' datetime.datetime(2021, 6, 19, 0, 0)
 '2021.11/2022.01' '未上市' '2021/6' datetime.datetime(2021, 6, 17, 0, 0)
 datetime.datetime(2021, 6, 21, 0, 0) datetime.datetime(2021, 6, 23, 0, 0)
 datetime.datetime(2021, 6, 28, 0, 0) datetime.datetime(2021, 6, 29, 0, 0)
 datetime.datetime(2021, 7, 1, 0, 0) datetime.datetime(2021, 7, 9, 0, 0)
 datetime.datetime(2021, 7, 13, 0, 0) datetime.datetime(2021, 8, 6, 0, 0)
 datetime.datetime(2021, 8, 2, 0, 0) datetime.datetime(2021, 7, 27, 0, 0)
 datetime.datetime(2021, 7, 17, 0, 0) datetime.datetime(2021, 7, 8, 0, 0)
 datetime.datetime(2021, 8, 29, 0, 0) datetime.datetime(2021, 8, 20, 0, 0)
 datetime.datetime(2021, 8, 14, 0, 0) datetime.datetime(2021, 9, 15, 0, 0)
 datetime.datetime(2021, 9, 13, 0, 0) datetime.datetime(2021, 9, 12, 0, 0)
 datetime.datetime(2021, 9, 7, 0, 0) datetime.datetime(2021, 9, 3, 0, 0)

In [21]:
def handle_year(col):
    if pd.api.types.is_datetime64_any_dtype(col):
        years = col.dt.year
    else:
        years = col.astype(str).str.extract(r'(\d{4})')[0].astype(float)
    
    # Fill NaN with a default value (e.g., 0 or median), then convert to int
    years = years.fillna(0).astype(int)
    return years

df['year'] = handle_year(df['launch_date'])


In [22]:
df.to_excel('../data/raw/rough_machining.xlsx',sheet_name='produced_by_py')